# Are the probe's libvmaf features redundant with its VMAF score?

The probe's VMAF score is computed from features that libvmaf also writes to its log
(`data/probe_73.csv`). Do those features add anything, or are they redundant with the score?

VMAF v1 uses only **4** of the 14 logged metrics (CAMBI, chroma SpEED uv, ADM3, motion3).
The other **10** are side outputs it never uses.

This notebook trains 7 models that differ only in their features and compares their errors.
It runs from `data/` alone. Expect about an hour on a 12-core machine.

In [1]:
import os
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.isotonic import IsotonicRegression

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'dense_vmaf_73.csv').exists())
DATA = Path(os.environ.get('VMAF_DATA', ROOT / 'data'))
OUT = Path(os.environ.get('VMAF_OUT', ROOT / 'out')); OUT.mkdir(exist_ok=True)

# ---- targets: 73 sources x 10 presets x 35 CRFs, VMAF v1 ----
d = pd.read_csv(DATA / 'dense_vmaf_73.csv').sort_values(['source', 'preset', 'crf']).reset_index(drop=True)
y, src, crf, pr = d.vmaf_v1.to_numpy(), d.source.to_numpy(), d.crf.to_numpy(), d.preset.to_numpy()

# ---- base features: probe VMAF and bitrate interpolated to the target CRF ----
lv = pd.read_csv(DATA / 'probe_73.csv').set_index(['source', 'probe_crf'])
a, b = lv.loc[[(s, 20) for s in src]], lv.loc[[(s, 51) for s in src]]
f = (crf - 20) / 31
qa, qb = np.sqrt(np.maximum(0, 100 - a.vmaf_mean.to_numpy())), np.sqrt(np.maximum(0, 100 - b.vmaf_mean.to_numpy()))
q = 100 - np.maximum(0, qa + (qb - qa) * f) ** 2
la, lb = np.log10(a.bitrate_kbps.to_numpy()), np.log10(b.bitrate_kbps.to_numpy())
rate = la + (lb - la) * np.clip(f, 0, 1)
g3 = pd.read_csv(DATA / 'good3_73.csv').set_index('source')
F = {'q': q, 'rate': rate, 'crf': crf, 'preset': pr}
for k in ['luma_mean', 'ti_mean', 'luma_std']:
    F[k] = g3.loc[src, k].to_numpy()

# ---- the 14 libvmaf metrics (pooled mean) at both probes ----
METRICS = ['motion_sad', 'cambi', 'adm2', 'adm3', 'adm_scale0', 'adm_scale1', 'adm_scale2', 'adm_scale3',
           'aim', 'motion2', 'motion3', 'speed_chroma_u', 'speed_chroma_uv', 'speed_chroma_v']   # libvmaf log order
VMAF_INPUTS = ['cambi', 'speed_chroma_uv', 'adm3', 'motion3']
ADM = ['adm2', 'adm_scale0', 'adm_scale1', 'adm_scale2', 'adm_scale3', 'aim']
for c in (20, 51):
    for m in METRICS:
        F[f'{m}@{c}'] = lv.loc[[(s, c) for s in src], f'{m}_mean'].to_numpy()
LV = [f'{m}@{c}' for c in (20, 51) for m in METRICS]

# ---- folds: one source held out at a time (Life and Life_v2 are near-copies, held out together) ----
groups = np.where(np.char.startswith(src.astype(str), 'Life_1080p30'), 'Life_1080p30', src)
# scored cells: everything except the anchor cell (p8/c40) and the wall cell (p10/c55)
anchor_cell = (pr == 8) & (crf == 40)
scored = ~(anchor_cell | ((pr == 10) & (crf == 55)))
anchor_p10 = (pr == 10) & (crf == 40)          # cheaper alternative anchor: full resolution at preset 10
scored_p10 = scored & ~anchor_p10              # for comparing the two anchors on exactly the same cells
print(len(d), 'cells,', len(set(groups)), 'folds,', scored.sum(), 'scored')

25550 cells, 72 folds, 25404 scored


**The 7 models.** Every model also gets the target CRF and preset.

| model | features |
|---|---|
| old method | probe VMAF + bitrate + GOOD-3 (3 simple content features) |
| base | probe VMAF + bitrate |
| base + 4 VMAF inputs | base + the 4 metrics VMAF is built from |
| base + 10 side outputs | base + the 10 metrics VMAF does not use |
| full | base + all 14 metrics (the current best model) |
| full without probe VMAF | full minus the probe VMAF score |
| full without ADM | full minus the 6 ADM side outputs |

Each model is scored in three settings:
- **under the wall**: the prediction as it comes out of the model.
- **anchored**: plus one full-resolution encode (preset 8, CRF 40). The whole predicted curve is shifted by that
  encode's error.
- **anchored at preset 10**: the same, but the full-resolution encode uses preset 10 (faster to encode).

In [2]:
BASE = ['q', 'rate', 'crf', 'preset']
MODELS = {
    'old method':              ['luma_mean', 'ti_mean', 'luma_std'] + BASE,
    'base':                    BASE,
    'base + 4 VMAF inputs':    BASE + [c for c in LV if c.split('@')[0] in VMAF_INPUTS],
    'base + 10 side outputs':  BASE + [c for c in LV if c.split('@')[0] not in VMAF_INPUTS],
    'full':                    BASE + LV,
    'full without probe VMAF': [c for c in BASE + LV if c != 'q'],
    'full without ADM':        BASE + [c for c in LV if c.split('@')[0] not in ADM],
}

def predict(cols):
    X = np.column_stack([F[c] for c in cols]); out = np.empty(len(y))
    for gname in sorted(set(groups)):
        te, tr = groups == gname, groups != gname
        m = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=1, max_features=1.0,
                                random_state=0, n_jobs=-1).fit(X[tr], y[tr])
        pte = np.clip(m.predict(X[te]), 0, 100)
        s_te, p_te, c_te = src[te], pr[te], crf[te]
        for s in set(s_te):                      # VMAF must not rise as CRF rises
            for k in range(1, 11):
                ix = (s_te == s) & (p_te == k)
                pte[ix] = IsotonicRegression(increasing=False).fit_transform(c_te[ix], pte[ix])
        out[te] = pte
    return out

def anchored(pred, cell=anchor_cell):
    shift = pd.Series((y - pred)[cell], index=src[cell])
    return np.clip(pred + shift.loc[src].to_numpy(), 0, 100)

pred = {}
for name, cols in MODELS.items():
    u = predict(cols)
    pred[name] = {'under the wall': u, 'anchored': anchored(u), 'anchored at preset 10': anchored(u, anchor_p10)}
    print(f"{name:25s} {len(cols):2d} features   MAE under the wall {abs(u - y)[scored].mean():.4f}   "
          f"anchored {abs(pred[name]['anchored'] - y)[scored].mean():.4f}")

old method                 7 features   MAE under the wall 1.7347   anchored 1.0722


base                       4 features   MAE under the wall 2.1627   anchored 1.3854


base + 4 VMAF inputs      12 features   MAE under the wall 1.7250   anchored 1.0345


base + 10 side outputs    24 features   MAE under the wall 1.6931   anchored 0.9641


full                      32 features   MAE under the wall 1.6811   anchored 0.9589


full without probe VMAF   31 features   MAE under the wall 2.0314   anchored 1.0859


full without ADM          20 features   MAE under the wall 1.7369   anchored 1.0361


## Results

**Change** = MAE of the first model minus MAE of the second; negative means the first is better.
The 95% interval comes from resampling the 72 held-out groups. If it does not cross 0, the difference is real.

In [3]:
rng = np.random.default_rng(20260930)
ug = sorted(set(groups)); draws = rng.integers(len(ug), size=(20000, len(ug)))
gidx = pd.Series(range(len(ug)), index=ug).loc[groups[scored]].to_numpy()
cnt = np.bincount(gidx, minlength=len(ug))

def compare(m1, m2, setting):
    dlt = abs(pred[m1][setting] - y)[scored] - abs(pred[m2][setting] - y)[scored]
    sums = np.bincount(gidx, weights=dlt, minlength=len(ug))
    boot = sums[draws].sum(1) / cnt[draws].sum(1)
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return dlt.mean(), lo, hi

QUESTIONS = [
    ('Do the 4 VMAF inputs add to the probe VMAF score?', 'base + 4 VMAF inputs', 'base'),
    ('Do the 10 side outputs add beyond the 4 VMAF inputs?', 'full', 'base + 4 VMAF inputs'),
    ('Is the probe VMAF score still needed with all 14 metrics?', 'full without probe VMAF', 'full'),
    ('Are the 4 VMAF inputs needed once the 10 side outputs are in?', 'base + 10 side outputs', 'full'),
    ('Are the ADM side outputs needed?', 'full without ADM', 'full'),
    ('Is the full model better than the old method?', 'full', 'old method'),
]
rows = []
for qtext, m1, m2 in QUESTIONS:
    r = {'question': qtext, 'comparison': f'{m1}  vs  {m2}'}
    for st in ['under the wall', 'anchored']:
        dl, lo, hi = compare(m1, m2, st)
        r[f'{st}: change'] = round(dl, 3)
        r[f'{st}: 95% interval'] = f'[{lo:+.3f}, {hi:+.3f}]'
        r[f'{st}: real?'] = 'yes' if hi < 0 or lo > 0 else 'no'
    rows.append(r)
# the preset-10 anchor against the preset-8 anchor, full model, on the same cells (both anchor cells excluded)
gidx10 = pd.Series(range(len(ug)), index=ug).loc[groups[scored_p10]].to_numpy()
dlt = abs(pred['full']['anchored at preset 10'] - y)[scored_p10] - abs(pred['full']['anchored'] - y)[scored_p10]
boot = np.bincount(gidx10, weights=dlt, minlength=len(ug))[draws].sum(1) / np.bincount(gidx10, minlength=len(ug))[draws].sum(1)
lo, hi = np.quantile(boot, [0.025, 0.975])
rows.append({'question': 'Can the anchor use preset 10 instead of preset 8?',
             'comparison': 'full, anchor at preset 10  vs  full, anchor at preset 8',
             'under the wall: change': None, 'under the wall: 95% interval': '', 'under the wall: real?': '',
             'anchored: change': round(dlt.mean(), 3), 'anchored: 95% interval': f'[{lo:+.3f}, {hi:+.3f}]',
             'anchored: real?': 'yes' if hi < 0 or lo > 0 else 'no'})
res = pd.DataFrame(rows)
mae = pd.DataFrame({st: {n: abs(pred[n][st] - y)[scored].mean() for n in MODELS} for st in ['under the wall', 'anchored']})
mae['anchored at preset 10'] = {n: abs(pred[n]['anchored at preset 10'] - y)[scored_p10].mean() for n in MODELS}
mae = mae.round(4)
mae.to_csv(OUT / 'libvmaf_ablation73_mae.csv'); res.to_csv(OUT / 'libvmaf_ablation73.csv', index=False)
display(mae)
display(res.set_index('question'))

,under the wall,anchored,anchored at preset 10
old method,1.7347,1.0722,1.3602
base,2.1627,1.3854,1.6657
base + 4 VMAF inputs,1.7250,1.0345,1.3260
base + 10 side outputs,1.6931,0.9641,1.2590
full,1.6811,0.9589,1.2426
full without probe VMAF,2.0314,1.0859,1.3538
full without ADM,1.7369,1.0361,1.3154


,comparison,under the wall: change,under the wall: 95% interval,under the wall: real?,anchored: change,anchored: 95% interval,anchored: real?
question,,,,,,,
Do the 4 VMAF inputs add to the probe VMAF score?,base + 4 VMAF inputs vs base,-0.438,"[-0.632, -0.256]",yes,-0.351,"[-0.498, -0.221]",yes
Do the 10 side outputs add beyond the 4 VMAF inputs?,full vs base + 4 VMAF inputs,-0.044,"[-0.126, +0.038]",no,-0.076,"[-0.113, -0.039]",yes
Is the probe VMAF score still needed with all 14 metrics?,full without probe VMAF vs full,0.350,"[+0.074, +0.622]",yes,0.127,"[+0.033, +0.227]",yes
Are the 4 VMAF inputs needed once the 10 side outputs are in?,base + 10 side outputs vs full,0.012,"[-0.064, +0.088]",no,0.005,"[-0.022, +0.036]",no
Are the ADM side outputs needed?,full without ADM vs full,0.056,"[-0.014, +0.127]",no,0.077,"[+0.038, +0.116]",yes
Is the full model better than the old method?,full vs old method,-0.054,"[-0.195, +0.086]",no,-0.113,"[-0.185, -0.048]",yes
Can the anchor use preset 10 instead of preset 8?,"full, anchor at preset 10 vs full, anchor at...",NaN,,,0.283,"[+0.171, +0.409]",yes


## What this shows

1. **The VMAF score does not make its features redundant.** Adding the 4 features VMAF is built from cuts the
   error clearly, in both settings. VMAF squeezes them into one number per frame and loses most of their detail.
2. **The features do not make the VMAF score redundant either.** Removing the score while keeping all 14 metrics
   makes the error clearly worse. The two carry different information.
3. **The redundancy is among the features themselves.** Once the 10 side outputs are in, the 4 VMAF inputs add
   almost nothing. The ADM side outputs are the most useful group.
4. **Which features to use:**
   - **With the anchor**, use all 14 metrics. The side outputs add a real gain on top of the 4 VMAF inputs.
   - **Under the wall**, the 4 VMAF inputs alone do about as well as all 14.
5. **The extra features cost nothing.** They come from the same libvmaf pass that scores the probe.
6. **A preset-10 anchor is cheaper but less accurate.** It costs about 1.9x instead of 2.3x one normal encode plus
   its scoring (estimated from encode timings), but the error rises clearly (0.96 to 1.24 for the full model).
   It is still much better than no anchor (1.68).

Caveat: all numbers come from the 73 sources the method was developed on. Differences smaller than about 0.01 MAE
are noise.